# LangChain: Agents
## Outline:

* Using built in LangChain tools: DuckDuckGo search and Wikipedia
* Defining your own tools

In [ ]:
# ===== 导入必要的库和模块 =====

# 环境变量和警告处理
import os
import warnings
from dotenv import load_dotenv, find_dotenv

# 核心组件 - 使用新版本的 create_agent
from langchain.agents import create_agent  # 新版本的 Agent 构造函数
from langchain_openai import ChatOpenAI  # OpenAI Chat模型

# 工具定义 - 使用 @tool 装饰器
from langchain.tools import tool  # 新版本推荐使用 tool 装饰器

# 忽略警告
warnings.filterwarnings("ignore")

# 加载环境变量
_ = load_dotenv(find_dotenv())

# ===== 配置 =====
llm = ChatOpenAI(
    temperature=0.0, 
    model="qwen-max",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

# ===== 1. 定义工具 =====

# 数学计算工具 - 使用 @tool 装饰器定义
@tool
def calculate(expression: str) -> str:
    """执行数学计算。输入应该是一个数学表达式字符串。"""
    try:
        # 使用 eval 计算表达式（生产环境建议使用更安全的方法）
        result = eval(expression)
        return f"计算结果: {result}"
    except Exception as e:
        return f"计算错误: {str(e)}"

# Wikipedia 查询工具
@tool
def search_wikipedia(query: str) -> str:
    """在 Wikipedia 上搜索信息。"""
    # 这里简化实现，实际应该调用 Wikipedia API
    # 可以使用 langchain_community.utilities.WikipediaAPIWrapper
    import wikipedia
    try:
        # 设置语言为中文（可选）
        wikipedia.set_lang("en")
        # 搜索并获取页面摘要
        results = wikipedia.search(query, results=2)
        if not results:
            return f"未找到关于 '{query}' 的 Wikipedia 页面"
        
        # 获取第一个结果的摘要
        page = wikipedia.page(results[0], auto_suggest=False)
        return f"标题: {page.title}\n\n摘要: {page.summary[:500]}...\n\nURL: {page.url}"
    except wikipedia.exceptions.DisambiguationError as e:
        # 处理歧义页面
        return f"查询 '{query}' 有多个含义，请更具体。选项: {', '.join(e.options[:5])}"
    except wikipedia.exceptions.PageError:
        return f"未找到关于 '{query}' 的 Wikipedia 页面"
    except Exception as e:
        return f"Wikipedia 搜索出错: {str(e)}"
    
# ===== 2. 创建 Agent (新版本方式) =====

# 使用 create_agent 创建 Agent
agent = create_agent(
    model=llm,  # 传入模型实例
    tools=[calculate, search_wikipedia],  # 传入工具列表
    system_prompt="You are a powerful assistant. Use the tools provided to answer the user's question."
)


def stream_debug(question: str):
    # 使用 stream 方法进行流式输出
    for chunk in agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="debug"  # 获取完整状态更新
    ):
        # 打印最新消息
        print(chunk)


# ===== 3. 执行示例 =====

print("--- 1. 数学计算演示 ---")
math_question = "What is the 25% of 300?"
print(f"Question: {math_question}")

# debug流模式输出，可以看具体执行过程
stream_debug(math_question)
# 使用 invoke 方法执行，直接获得最终结果
# result_math = agent.invoke({
#     "messages": [{"role": "user", "content": math_question}]
# })
# # 获取最后一条消息的内容
# print(f"Result: {result_math['messages'][-1].content}\n")


print("--- 2. Wikipedia 查询演示 ---")
wiki_question = (
    "Tom M. Mitchell is an American computer scientist "
    "and the Founders University Professor at Carnegie Mellon University (CMU) "
    "what book did he write?"
)
print(f"Question: {wiki_question}")

# debug流模式输出，可以看具体执行过程
stream_debug(wiki_question)
# 使用 invoke 方法执行，直接获得最终结果
# result_wiki = agent.invoke({
#     "messages": [{"role": "user", "content": wiki_question}]
# })
# print(f"Result: {result_wiki['messages'][-1].content}\n")



--- 1. 数学计算演示 ---
Question: What is the 25% of 300?
{'step': 1, 'timestamp': '2025-11-05T10:04:45.524019+00:00', 'type': 'task', 'payload': {'id': '4bd0a76d-b0d9-8a2d-9a48-020298598999', 'name': 'model', 'input': {'messages': [HumanMessage(content='What is the 25% of 300?', additional_kwargs={}, response_metadata={}, id='a7d3ecca-7a2f-4cfc-9018-c5ebff930d24')]}, 'triggers': ('branch:to:model',)}}
{'step': 1, 'timestamp': '2025-11-05T10:04:47.506090+00:00', 'type': 'task_result', 'payload': {'id': '4bd0a76d-b0d9-8a2d-9a48-020298598999', 'name': 'model', 'error': None, 'result': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_41ac89b2ac424155963078', 'function': {'arguments': '{"expression": "0.25 * 300"}', 'name': 'calculate'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 318, 'total_tokens': 345, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_token

In [ ]:

# --- 2. Python Agent (使用 PythonREPLTool) ---

# 2.1 初始化 Python REPL Tool
# 推荐使用 from_tool_string 创建 PythonREPLTool，它是一个封装过的工具，更安全。
# 但为了保持和原始文件 L6-Agents.ipynb 的功能对齐，我们直接使用 PythonREPLTool()
python_tool = PythonREPLTool()
python_tools = [python_tool]

# 2.2 创建 Python Agent (Agent 逻辑不变，只是工具不同)
agent_runnable_2 = create_agent(llm, python_tools, prompt=prompt_template)
python_agent_executor = AgentExecutor(agent=agent_runnable_2, tools=python_tools, verbose=True)

# 2.3 客户列表数据
customer_list = [
    ["Harrison", "Chase"], 
    ["Lang", "Chain"],
    ["Dolly", "Too"],
    ["Elle", "Elem"], 
    ["Geoff","Fusion"], 
    ["Trance","Former"],
    ["Jen","Ayai"]
]

# 2.4 运行 Python Agent 排序任务
print("--- 2. Python Agent 演示: 列表排序 ---")
sort_prompt = (
    f"Sort these customers by last name and then first name and print the output: {customer_list}"
)
print(f"Instruction: {sort_prompt}")
result_sort = python_agent_executor.invoke({"input": sort_prompt})
print(f"Result: {result_sort['output']}\n")


# --- 3. 定义你自己的工具 (自定义日期工具) ---

# 3.1 使用 @tool 装饰器定义自定义工具
@tool
def time(text: str) -> str:
    """Returns todays date, use this for any 
    questions related to knowing todays date. 
    The input should always be an empty string, 
    and this function will always return todays 
    date - any date mathmatics should occur 
    outside this function."""
    return str(date.today())

# 3.2 组合所有工具 (内置 + 自定义)
all_tools = built_in_tools + [time]

# 3.3 创建新的 AgentExecutor
agent_runnable_3 = create_agent(llm, all_tools, prompt=prompt_template)
time_agent_executor = AgentExecutor(agent=agent_runnable_3, tools=all_tools, verbose=True)

# 3.4 运行自定义工具任务
print("--- 3. 自定义工具演示: 获取当前日期 ---")
date_question = "whats the date today?"
print(f"Question: {date_question}")

result_date = time_agent_executor.invoke({"input": date_question}) 
print(f"Result: {result_date['output']}")